# Airbnb Prices in European Cities — Auto PPTX Generator

**Run in Google Colab.** Before executing this notebook:
1. Upload all city CSV files (e.g. `amsterdam_weekdays.csv`, `amsterdam_weekends.csv`, …) to `/content`.
2. Run every cell top-to-bottom.
3. The finished presentation will be saved as `/content/Airbnb_Europe_Analysis.pptx` and downloaded automatically.

**Required packages:** `pandas`, `matplotlib`, `seaborn`, `python-pptx` (all installed below).

## 0 — Install dependencies

In [ ]:
!pip -q install python-pptx

## 1 — Load & concatenate all city CSV files

In [ ]:
import glob
import os
import pandas as pd

files = sorted(
    glob.glob("/content/*_weekdays.csv") +
    glob.glob("/content/*_weekends.csv")
)

if not files:
    raise FileNotFoundError(
        "No CSV files found in /content. "
        "Please upload *_weekdays.csv and *_weekends.csv files first."
    )

dfs = []
for f in files:
    base = os.path.basename(f).replace(".csv", "")  # e.g. amsterdam_weekdays
    city, day_type = base.rsplit("_", 1)            # city, weekdays/weekends
    tmp = pd.read_csv(f)
    tmp["city"] = city
    tmp["day_type"] = day_type
    dfs.append(tmp)

all_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(files)} files → combined shape: {all_df.shape}")
all_df.head()

## 2 — Clean columns

In [ ]:
# Standardise column names to lowercase
all_df.columns = [c.strip().lower() for c in all_df.columns]

# Drop spurious index column produced by some CSV exports
if "unnamed: 0" in all_df.columns:
    all_df = all_df.drop(columns=["unnamed: 0"])

print("Shape after cleaning:", all_df.shape)
print("Columns:", all_df.columns.tolist())
print("\nMissing values:")
print(all_df.isna().sum().sort_values(ascending=False).head(10))

## 3 — Summary tables

In [ ]:
# Average realsum by city
city_avg = (
    all_df.groupby("city")["realsum"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
city_avg_df = city_avg.reset_index()
city_avg_df.columns = ["city", "avg_realsum"]

print("Average realsum by city:")
print(city_avg)

# Average realsum by city and day_type
city_day_avg = (
    all_df.groupby(["city", "day_type"])["realsum"]
    .mean()
    .round(2)
    .reset_index()
    .sort_values(["city", "day_type"])
)
print("\nAverage realsum by city & day_type:")
print(city_day_avg)

# Average realsum by room_type
rt_avg = (
    all_df.groupby("room_type")["realsum"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
print("\nAverage realsum by room_type:")
print(rt_avg)

## 4 — Generate & save charts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
CHART_DIR = "/content"

# --- Chart 1: Average realsum by city (bar) ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=city_avg_df, x="city", y="avg_realsum", color="#4C72B0", ax=ax)
ax.set_title("Average Airbnb Price by City", fontsize=14, fontweight="bold")
ax.set_xlabel("City")
ax.set_ylabel("Average realsum (€)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
chart1 = os.path.join(CHART_DIR, "chart1_city_avg.png")
plt.savefig(chart1, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", chart1)

# --- Chart 2: Weekdays vs Weekends grouped bar ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=city_day_avg, x="city", y="realsum", hue="day_type", ax=ax)
ax.set_title("Weekdays vs Weekends: Average Price by City", fontsize=14, fontweight="bold")
ax.set_xlabel("City")
ax.set_ylabel("Average realsum (€)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
chart2 = os.path.join(CHART_DIR, "chart2_weekday_weekend.png")
plt.savefig(chart2, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", chart2)

# --- Chart 3: Boxplot realsum by room_type ---
fig, ax = plt.subplots(figsize=(8, 5))
# Cap outliers for readability
plot_df = all_df[all_df["realsum"] < all_df["realsum"].quantile(0.99)]
sns.boxplot(data=plot_df, x="room_type", y="realsum", ax=ax)
ax.set_title("Price Distribution by Room Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Room type")
ax.set_ylabel("realsum (€)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
chart3 = os.path.join(CHART_DIR, "chart3_room_type_box.png")
plt.savefig(chart3, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", chart3)

# --- Chart 4: Scatter realsum vs person_capacity (sampled) ---
sample_df = all_df.sample(min(8000, len(all_df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=sample_df, x="person_capacity", y="realsum",
                alpha=0.35, ax=ax)
ax.set_title("Price vs Person Capacity (sample)", fontsize=14, fontweight="bold")
ax.set_xlabel("Person capacity")
ax.set_ylabel("realsum (€)")
plt.tight_layout()
chart4 = os.path.join(CHART_DIR, "chart4_scatter_capacity.png")
plt.savefig(chart4, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", chart4)

# --- Chart 5: Line average realsum by person_capacity ---
cap_avg = (
    all_df.groupby("person_capacity")["realsum"]
    .mean()
    .reset_index()
    .sort_values("person_capacity")
)
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=cap_avg, x="person_capacity", y="realsum", marker="o", ax=ax)
ax.set_title("Average Price by Person Capacity", fontsize=14, fontweight="bold")
ax.set_xlabel("Person capacity")
ax.set_ylabel("Average realsum (€)")
plt.tight_layout()
chart5 = os.path.join(CHART_DIR, "chart5_line_capacity.png")
plt.savefig(chart5, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", chart5)

## 5 — Build the PowerPoint presentation

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

# ── helpers ──────────────────────────────────────────────────────────────────

def _set_title_style(shape, size=28, bold=True, color=RGBColor(0x1F, 0x49, 0x7D)):
    """Apply consistent title formatting."""
    tf = shape.text_frame
    for para in tf.paragraphs:
        for run in para.runs:
            run.font.size = Pt(size)
            run.font.bold = bold
            run.font.color.rgb = color


def add_title_slide(prs, title, subtitle=""):
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    slide.shapes.title.text = title
    slide.placeholders[1].text = subtitle
    _set_title_style(slide.shapes.title, size=36)
    return slide


def add_bullets_slide(prs, title, bullets):
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = title
    _set_title_style(slide.shapes.title)
    tf = slide.placeholders[1].text_frame
    tf.clear()
    for i, bullet in enumerate(bullets):
        para = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        para.text = bullet
        para.level = 0
        para.runs[0].font.size = Pt(16)
    return slide


def add_image_slide(prs, title, img_path, caption=""):
    slide = prs.slides.add_slide(prs.slide_layouts[5])  # title-only layout
    slide.shapes.title.text = title
    _set_title_style(slide.shapes.title)

    # Image: centred below title
    img_top = Inches(1.4)
    img_height = Inches(4.8)
    pic = slide.shapes.add_picture(img_path, Inches(0.8), img_top, height=img_height)

    # Optional caption at the bottom
    if caption:
        txBox = slide.shapes.add_textbox(Inches(0.5), Inches(6.4), Inches(9), Inches(0.5))
        tf = txBox.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = caption
        p.alignment = PP_ALIGN.CENTER
        p.runs[0].font.size = Pt(12)
        p.runs[0].font.italic = True
        p.runs[0].font.color.rgb = RGBColor(0x44, 0x44, 0x44)
    return slide


# ── build presentation ────────────────────────────────────────────────────────

prs = Presentation()
prs.slide_width  = Inches(10)
prs.slide_height = Inches(7.5)

# Slide 1 — Title
add_title_slide(
    prs,
    "Airbnb Prices in European Cities",
    "Data Manipulation & Visualization Analysis\nGoogle Colab · python-pptx"
)

# Slide 2 — Dataset & Goal
add_bullets_slide(
    prs,
    "Dataset & Goal",
    [
        "📂  Dataset: Airbnb prices across 10 European cities (weekdays & weekends)",
        "📊  51,707 listings | 21 features including price, room type, capacity, location",
        "🎯  Goal: Understand pricing differences and key pricing drivers",
        "🔧  Methods: Data cleaning → feature engineering → group analysis → visualisation",
        "🏙️  Cities: Amsterdam · Athens · Barcelona · Berlin · Budapest",
        "        Lisbon · London · Paris · Rome · Vienna",
    ]
)

# Slide 3 — Chart: avg price by city
add_image_slide(
    prs,
    "Average Airbnb Price by City",
    chart1,
    caption="Amsterdam leads at ~€573; Athens is the most affordable at ~€152."
)

# Slide 4 — Chart: weekdays vs weekends
add_image_slide(
    prs,
    "Weekdays vs Weekends: Average Price by City",
    chart2,
    caption="Weekends are higher in most cities; Paris is a notable exception (weekdays > weekends)."
)

# Slide 5 — Chart: price by room type boxplot
add_image_slide(
    prs,
    "Price Distribution by Room Type",
    chart3,
    caption="Entire home/apt has the highest median price and widest spread; shared rooms are cheapest."
)

# Slide 6 — Chart: scatter price vs capacity
add_image_slide(
    prs,
    "Price vs Person Capacity",
    chart4,
    caption="A positive relationship exists between capacity and price, with significant spread per capacity level."
)

# Slide 7 — Chart: line avg price by capacity
add_image_slide(
    prs,
    "Average Price by Person Capacity (Trend)",
    chart5,
    caption="Average price rises steadily with capacity, confirming larger listings command higher rates."
)

# Slide 8 — Key Findings
top_city = city_avg_df.iloc[0]["city"].capitalize()
top_price = city_avg_df.iloc[0]["avg_realsum"]
low_city  = city_avg_df.iloc[-1]["city"].capitalize()
low_price = city_avg_df.iloc[-1]["avg_realsum"]
top_rt    = rt_avg.index[0]
top_rt_p  = rt_avg.iloc[0]

add_bullets_slide(
    prs,
    "Key Findings",
    [
        f"🏆  {top_city} has the highest average price (~€{top_price:.0f}); "
        f"{low_city} is the most affordable (~€{low_price:.0f})",
        "📅  Weekend prices exceed weekday prices in most cities (Amsterdam, Barcelona, Berlin…)",
        "     Exception: Paris shows slightly higher weekday prices",
        f"🛏️  '{top_rt}' commands the highest average price (~€{top_rt_p:.0f})",
        "     Shared rooms are the cheapest option across all cities",
        "👥  Listings with higher person capacity consistently cost more",
        "📍  Location (dist, metro_dist) and cleanliness ratings also correlate with price",
    ]
)

# Slide 9 — Conclusions & Next Steps
add_bullets_slide(
    prs,
    "Conclusions & Next Steps",
    [
        "✅  City, room type, and capacity are the strongest price drivers",
        "✅  Seasonality (weekdays vs weekends) adds a measurable premium in most cities",
        "✅  Superhosts and high cleanliness ratings correlate with premium pricing",
        "",
        "🔮  Next steps:",
        "     • Build a regression/ML model to predict listing prices",
        "     • Add time-series data to track seasonal trends",
        "     • Analyse impact of distance-to-city-centre on price per city",
        "     • Compare occupancy rates alongside prices for ROI analysis",
    ]
)

# ── save ──────────────────────────────────────────────────────────────────────
OUTPUT_PATH = "/content/Airbnb_Europe_Analysis.pptx"
prs.save(OUTPUT_PATH)
print(f"✅  Presentation saved to: {OUTPUT_PATH}")

## 6 — Download the PPTX

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print("Download triggered!")
except ImportError:
    print(f"Not running in Colab. Find your file at: {OUTPUT_PATH}")